# weld-qml-04-learning-curve

**Question it answers.** Early QML papers suggested quantum kernels help most when training data is scarce. This notebook tests that directly: the classifier heads are retrained on **1/8, 1/4, 1/2 and all** of each split's training radiographs (whole radiographs are dropped, never single sections), and quantum is compared with the classical RBF SVM at every size, on the same 25 splits and the same real-only test sets. No cut-paste images are used anywhere. Each smaller size is drawn **3 times** with different radiographs and averaged, to reduce noise.

**The main test (fixed before seeing any results).** For each feature set and quantum head: the quantum − RBF difference in test AUC, averaged over the three reduced sizes (1/8, 1/4, 1/2), with a 95% confidence interval from the corrected repeated-CV t-test. If the interval lies above 0, quantum is better with little data. If it lies below 0, quantum is worse. Otherwise the notebook reports the largest advantage the data cannot rule out. It also flags sizes where even the classical model is not above chance, because there "no difference" just means nothing learns.

It also checks how the **geometric difference** *g* between the quantum and classical kernels depends on the smoothing setting λ used to compute it.

Heads: SVM (RBF), QSVM (ZZ map), QSVM (no entanglement), projected quantum kernel. Qubits d ∈ {4, 8} chosen on validation; bandwidth/γ and C tuned on validation exactly as in notebook 02.

Note: the fine-tuned ResNet features come from CNNs trained on the full training set, so for those two feature sets only the *classifier head* sees less data. HOG and frozen-ResNet features do not depend on the training data, so they are the cleanest test.

**Settings:** CPU, Internet off. **Input:** the output of notebook 01 (`weld-qml-01-cnn-embeddings`).
**Outputs:** `learning_curve.csv`, `geo_lambda.csv`, `tables/table9_learning_curve.csv` (+ .tex), `tables/table10_small_data_test.csv` (+ .tex), `figures/figD_learning_curve.pdf/.png`.
**Runtime:** about 45–75 min. Progress is saved after every split; if the session stops, add this notebook's output as an extra input and run again to resume.

In [ ]:
import builtins, functools, os, glob, time, numpy as np, pandas as pd, matplotlib
print = functools.partial(builtins.print, flush=True)
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from itertools import combinations
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else None

META = find("qml_meta.csv")
assert META, "Add the output of notebook 01 (weld-qml-01-cnn-embeddings) as an input"
IN = os.path.dirname(META); OUT = "/kaggle/working"
os.makedirs(f"{OUT}/tables", exist_ok=True); os.makedirs(f"{OUT}/figures", exist_ok=True)
SEED = 42; ZZ_REPS, ALPHA = 2, 2.0; GEO_LAMBDA = 0.1
SOURCES = ["hog", "frozen", "cnnreal", "cnncp"]
FRACS = [0.125, 0.25, 0.5, 1.0]
DIMS = [4, 8]
BW = np.geomspace(0.005, 2.0, 10)
GAMMA_D = np.geomspace(0.003, 10.0, 10)
CS = [0.1, 1.0, 10.0, 100.0]
LAMBDAS = [1.0, 0.1, 0.01, 0.001]
MIN_PER_CLASS = 5
N_DRAWS = 3

In [ ]:
def _bits(n):
    return ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1).astype(np.float64)

def _wht(psi):
    m, N = psi.shape; h = 1
    while h < N:
        v = psi.reshape(m, N // (2 * h), 2, h)
        psi = np.stack([v[:, :, 0] + v[:, :, 1], v[:, :, 0] - v[:, :, 1]], axis=2).reshape(m, N)
        h *= 2
    return psi / np.sqrt(N)

def zz_states(X, reps=ZZ_REPS, alpha=ALPHA):
    X = np.asarray(X, np.float64); m, n = X.shape; N = 2 ** n; Z = _bits(n)
    ph = X @ Z.T
    if n > 1:
        P = list(combinations(range(n), 2))
        A = np.stack([(np.pi - X[:, i]) * (np.pi - X[:, j]) for i, j in P], 1)
        Q = np.stack([np.logical_xor(Z[:, i], Z[:, j]) for i, j in P], 1).astype(np.float64)
        ph = ph + A @ Q.T
    D = np.exp(1j * alpha * ph); psi = D / np.sqrt(N)
    for _ in range(reps - 1): psi = D * _wht(psi)
    return psi

def fidelity(Sa, Sb): return np.abs(Sa @ Sb.conj().T) ** 2

def prod_kernel(A, B, reps=ZZ_REPS, alpha=ALPHA):
    def qs(X):
        e = np.exp(1j * alpha * X); a = np.full(X.shape, 1 / np.sqrt(2), complex); b = e / np.sqrt(2)
        for _ in range(reps - 1): a, b = (a + b) / np.sqrt(2), e * (a - b) / np.sqrt(2)
        return a, b
    a1, b1 = qs(A); a2, b2 = qs(B); K = np.ones((len(A), len(B)))
    for i in range(A.shape[1]):
        K *= np.abs(np.outer(a1[:, i].conj(), a2[:, i]) + np.outer(b1[:, i].conj(), b2[:, i])) ** 2
    return K

def bloch(S):
    m, N = S.shape; n = int(round(np.log2(N))); out = np.empty((m, n, 3))
    for i in range(n):
        v = S.reshape(m, N >> (i + 1), 2, 1 << i); a, b = v[:, :, 0, :], v[:, :, 1, :]
        c = np.einsum("mij,mij->m", a.conj(), b)
        out[:, i, 0], out[:, i, 1] = 2 * c.real, 2 * c.imag
        out[:, i, 2] = np.einsum("mij,mij->m", a.conj(), a).real - np.einsum("mij,mij->m", b.conj(), b).real
    return out.reshape(m, 3 * n)

def sqdist(A, B):
    return np.maximum((A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - 2 * A @ B.T, 0)

def best_threshold(yt, p):
    ts = np.unique(np.concatenate([p, [np.median(p)]]))
    ba = [balanced_accuracy_score(yt, (p >= t).astype(int)) for t in ts]
    return float(ts[int(np.argmax(ba))])

def test_metrics(yva, sva, yte, ste):
    thr = best_threshold(yva, sva); yp = (ste >= thr).astype(int)
    rec = recall_score(yte, yp, average=None, labels=[0, 1], zero_division=0)
    return dict(auc=roc_auc_score(yte, ste), bal_acc=balanced_accuracy_score(yte, yp), rec_def=rec[1], rec_clean=rec[0], thr=thr)

def fit_svm(Ktr, Kva, Kte, ytr, yva, Cs):
    best = None
    for C in Cs:
        m = SVC(kernel="precomputed", C=C, class_weight="balanced").fit(Ktr, ytr)
        sva = m.decision_function(Kva); a = roc_auc_score(yva, sva)
        if best is None or a > best[0] + 1e-12: best = (a, C, m, sva)
    a, C, m, sva = best
    return a, C, sva, m.decision_function(Kte)

def kstats(K, y):
    n = len(K); off = K[~np.eye(n, dtype=bool)]
    Kc = K - K.mean(0) - K.mean(1)[:, None] + K.mean()
    s = np.where(y == 1, 1.0, -1.0); s = s - s.mean(); Y = np.outer(s, s)
    return dict(off_mean=off.mean(), off_std=off.std(), kta=(Kc * Y).sum() / np.sqrt((Kc * Kc).sum() * (Y * Y).sum()))

def geo_diff(Kc, Kq, lam=GEO_LAMBDA):
    wq, Vq = np.linalg.eigh(Kq); sq = (Vq * np.sqrt(np.clip(wq, 0, None))) @ Vq.T
    wc, Vc = np.linalg.eigh(Kc); wc = np.clip(wc, 0, None)
    M = sq @ ((Vc * (wc / (wc + lam) ** 2)) @ Vc.T) @ sq
    return float(np.sqrt(max(np.linalg.eigvalsh(M).max(), 0)))

def prep_features(Xr_raw, Xc_raw, fit, d):
    sc = StandardScaler().fit(Xr_raw[fit])
    if d == 0:
        return sc.transform(Xr_raw), sc.transform(Xc_raw), 1.0
    p = PCA(d, random_state=SEED).fit(sc.transform(Xr_raw[fit]))
    sc2 = StandardScaler().fit(p.transform(sc.transform(Xr_raw[fit])))
    f = lambda X: sc2.transform(p.transform(sc.transform(X)))
    return f(Xr_raw), f(Xc_raw), float(p.explained_variance_ratio_.sum())

def tuned_head(name, S, tr, val, te, y):
    if name == "svm_rbf":
        D2 = sqdist(S, S); ks = [(g, np.exp(-(g / S.shape[1]) * D2)) for g in GAMMA_D]
    elif name == "qsvm_zz":
        ks = []
        for c in BW: SV = zz_states(c * S); ks.append((c, fidelity(SV, SV)))
    elif name == "qsvm_prod":
        ks = [(c, prod_kernel(c * S, c * S)) for c in BW]
    else:
        ks = []
        for c in BW:
            B = bloch(zz_states(c * S)); Db = sqdist(B, B); gb = 1.0 / max(np.mean(Db[np.ix_(tr, tr)]), 1e-12); ks.append((c, np.exp(-gb * Db)))
    best = None
    for bw, K in ks:
        a, C, sva, ste = fit_svm(K[np.ix_(tr, tr)], K[np.ix_(val, tr)], K[np.ix_(te, tr)], y[tr], y[val], CS)
        if best is None or a > best[0] + 1e-12: best = (a, C, bw, sva, ste, K)
    return best

In [ ]:
meta = pd.read_csv(META); yr = meta.label.values; gr = meta.image.values
H_real = np.load(os.path.join(IN, "hog_real.npy")).astype(np.float64)
F_real = np.load(os.path.join(IN, "frozen_real.npy")).astype(np.float64)
EMB = sorted(glob.glob(os.path.join(IN, "emb", "emb_r*_f*.npz")))
print(f"{len(yr)} sections | {len(EMB)} split files")
prev = find("learning_curve.csv")
rows = pd.read_csv(prev).to_dict("records") if prev and os.path.dirname(prev) != IN else []
done = {(r["repeat"], r["fold"]) for r in rows}
print("splits already done:", len(done))

In [ ]:
T0 = time.time()
for path in EMB:
    z = np.load(path); rep, k = map(int, os.path.basename(path)[5:-4].replace("r", "").split("_f"))
    if (rep, k) in done: continue
    fit, val, te = z["fit"], z["val"], z["te"]
    feats = {"hog": H_real, "frozen": F_real, "cnnreal": z["cnnreal_real"].astype(np.float64), "cnncp": z["cnncp_real"].astype(np.float64)}
    imgs = np.array(sorted(set(gr[fit])))
    for f in FRACS:
      for j in range(N_DRAWS if f < 1 else 1):
        r = np.random.default_rng(1000 + 100 * rep + k + 7919 * j)
        for _ in range(50):
            pick = r.choice(imgs, max(1, int(round(f * len(imgs)))), replace=False)
            sub = fit[np.isin(gr[fit], pick)]
            if min((yr[sub] == 0).sum(), (yr[sub] == 1).sum()) >= MIN_PER_CLASS: break
        for src in SOURCES:
            for d in DIMS:
                Zr, _, _ = prep_features(feats[src], feats[src][:2], sub, d)
                for m in ["svm_rbf", "qsvm_zz", "qsvm_prod", "pqk"]:
                    a, C, bw, sva, ste, K = tuned_head(m, Zr, sub, val, te, yr)
                    rows.append(dict(repeat=rep, fold=k, frac=f, draw=j, n_radiographs=len(pick), n_train=len(sub), source=src, d=d, model=m,
                                     bw=bw, C=C, val_auc=a, auc=roc_auc_score(yr[te], ste)))
    pd.DataFrame(rows).to_csv(f"{OUT}/learning_curve.csv", index=False)
    print(f"rep {rep} fold {k} done | {(time.time()-T0)/60:.1f} min")

## Geometric difference vs the smoothing setting λ (d = 8, all training data)

In [ ]:
geo = []
for path in EMB:
    z = np.load(path); rep, k = map(int, os.path.basename(path)[5:-4].replace("r", "").split("_f"))
    fit, val, te = z["fit"], z["val"], z["te"]
    feats = {"hog": H_real, "frozen": F_real, "cnnreal": z["cnnreal_real"].astype(np.float64), "cnncp": z["cnncp_real"].astype(np.float64)}
    for src in SOURCES:
        Zr, _, _ = prep_features(feats[src], feats[src][:2], fit, 8)
        Kc = tuned_head("svm_rbf", Zr, fit, val, te, yr)[5][np.ix_(fit, fit)]
        for m in ["qsvm_zz", "pqk"]:
            Kq = tuned_head(m, Zr, fit, val, te, yr)[5][np.ix_(fit, fit)]
            for lam in LAMBDAS:
                geo.append(dict(repeat=rep, fold=k, source=src, model=m, lam=lam, g=geo_diff(Kc, Kq, lam), sqrtN=np.sqrt(len(fit))))
G = pd.DataFrame(geo); G.to_csv(f"{OUT}/geo_lambda.csv", index=False)
print("median geometric difference g (rows) by smoothing λ (columns); sqrt(N) =", round(G.sqrtN.median(), 1))
print(G.pivot_table(index=["source", "model"], columns="lam", values="g", aggfunc="median").round(2))

## Table 9 and Figure D — accuracy vs amount of training data

In [ ]:
SRC = {"hog": "HOG", "frozen": "ResNet-18 frozen", "cnnreal": "ResNet-18 fine-tuned (real)", "cnncp": "ResNet-18 fine-tuned (+cut-paste)"}
MOD = {"svm_rbf": "SVM (RBF)", "qsvm_zz": "QSVM (ZZ map)", "qsvm_prod": "QSVM (no entanglement)", "pqk": "Projected quantum kernel"}
K = ["repeat", "fold"]
L = pd.DataFrame(rows)
if "draw" not in L: L["draw"] = 0
LT = L.loc[L.groupby(K + ["frac", "draw", "source", "model"]).val_auc.idxmax()]
LT = LT.groupby(K + ["frac", "source", "model"]).agg(auc=("auc", "mean"), n_radiographs=("n_radiographs", "median"),
                                                     n_train=("n_train", "median")).reset_index()

def corrected(a, b, folds=5):
    d = (a - b).dropna(); n = len(d); m = d.mean(); se = np.sqrt((1 / n + 1 / (folds - 1)) * d.var(ddof=1))
    h = stats.t.ppf(0.975, n - 1) * se
    return m, m - h, m + h, int((d > 0).sum()), n

out = []
for (s, f), g in LT.groupby(["source", "frac"]):
    p = g.pivot_table(index=K, columns="model", values="auc")
    row = dict(features=SRC[s], fraction=f, radiographs=int(g.n_radiographs.median()), sections=int(g.n_train.median()),
               **{MOD[m]: f"{p[m].mean():.3f}" for m in MOD})
    for q in ["qsvm_zz", "qsvm_prod", "pqk"]:
        m, lo, hi, w, n = corrected(p[q], p["svm_rbf"])
        row[f"{MOD[q]} − RBF"] = f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}] ({w}/{n})"
    out.append(row)
T9 = pd.DataFrame(out)
T9.to_csv(f"{OUT}/tables/table9_learning_curve.csv", index=False)
cols = ["features", "radiographs", "sections"] + list(MOD.values())
with open(f"{OUT}/tables/table9_learning_curve.tex", "w") as fh:
    fh.write("\\begin{table}[htbp]\n\\centering\n\\caption{Test AUC when the classifier is trained on a fraction of each split's training radiographs (qubits and hyperparameters chosen on validation; mean over 25 splits).}\n\\label{tab:qml_learning_curve}\n\\small\n")
    fh.write("\\begin{tabular}{l" + "c" * (len(cols) - 1) + "}\n\\toprule\n" + " & ".join(cols) + " \\\\\n\\midrule\n")
    for _, r in T9.iterrows(): fh.write(" & ".join(str(r[c]) for c in cols) + " \\\\\n")
    fh.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "axes.edgecolor": "#8a8984", "axes.linewidth": 0.6,
                     "axes.spines.top": False, "axes.spines.right": False, "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "legend.frameon": False, "savefig.bbox": "tight", "savefig.dpi": 300})
COL = {"svm_rbf": "#2a78d6", "qsvm_zz": "#eb6834", "qsvm_prod": "#1baf7a", "pqk": "#eda100"}
MRK = {"svm_rbf": "o", "qsvm_zz": "s", "qsvm_prod": "^", "pqk": "D"}
SHORT = {"hog": "HOG", "frozen": "ResNet-18\nfrozen", "cnnreal": "ResNet-18 fine-tuned\n(real)", "cnncp": "ResNet-18 fine-tuned\n(+cut-paste)"}
srcs = [s for s in SOURCES if s in set(LT.source)]
fig, axes = plt.subplots(1, len(srcs), figsize=(1.8 * len(srcs), 2.6), sharey=True); axes = np.atleast_1d(axes)
for ax, s in zip(axes, srcs):
    g = LT[LT.source == s]
    for j, m in enumerate(COL):
        st = g[g.model == m].groupby("frac").agg(n=("n_radiographs", "median"), mean=("auc", "mean"), se=("auc", lambda x: x.std(ddof=1) / np.sqrt(len(x))))
        x = st.n.values * (1 + (j - 1.5) * 0.04)
        ax.errorbar(x, st["mean"], yerr=st.se, color=COL[m], marker=MRK[m], ms=4, lw=1.4, elinewidth=0.8, capsize=0, mec="white", mew=0.8)
    ax.axhline(0.5, color="#8a8984", lw=0.8, ls=(0, (2, 2)))
    ax.set_xscale("log"); xt = sorted(g.groupby("frac").n_radiographs.median().unique()); ax.set_xticks(xt); ax.set_xticklabels([str(int(v)) for v in xt]); ax.minorticks_off()
    ax.set_title(SHORT[s], fontsize=8); ax.grid(axis="y", color="#e4e3df", lw=0.6); ax.set_axisbelow(True)
axes[0].set_ylabel("test AUC (mean ± s.e.)")
from matplotlib.lines import Line2D
fig.supxlabel("training radiographs used by the classifier (log scale)", fontsize=8, y=0.1)
fig.legend([Line2D([], [], color=COL[m], marker=MRK[m], ms=4, lw=1.4, mec="white") for m in COL] + [Line2D([], [], color="#8a8984", lw=0.8, ls=(0, (2, 2)))],
           [MOD[m] for m in COL] + ["chance"], loc="lower center", ncol=5, fontsize=7, bbox_to_anchor=(0.5, -0.06))
fig.subplots_adjust(bottom=0.26, wspace=0.12)
fig.savefig(f"{OUT}/figures/figD_learning_curve.pdf"); fig.savefig(f"{OUT}/figures/figD_learning_curve.png"); plt.close(fig)
print(T9.to_string(index=False))

## Table 10 — the direct test: is quantum better when training data is scarce?

In [ ]:
def ci1(d, folds=5):
    d = pd.Series(d).dropna(); n = len(d); m = d.mean(); se = np.sqrt((1 / n + 1 / (folds - 1)) * d.var(ddof=1))
    h = stats.t.ppf(0.975, n - 1) * se
    return m, m - h, m + h, int((d > 0).sum()), n

small_fracs = [f for f in FRACS if f < 1]
learn = []
for (s, f), g in LT[LT.model == "svm_rbf"].groupby(["source", "frac"]):
    m, lo, hi, _, _ = ci1(g.auc - 0.5)
    learn.append(dict(source=s, frac=f, rbf_auc=0.5 + m, above_chance=lo > 0))
learn = pd.DataFrame(learn)

out = []
for s in srcs:
    p = LT[LT.source == s].pivot_table(index=K + ["frac"], columns="model", values="auc")
    for q in ["qsvm_zz", "qsvm_prod", "pqk"]:
        gap = (p[q] - p["svm_rbf"]).unstack("frac")
        m, lo, hi, w, n = ci1(gap[small_fracs].mean(axis=1))
        mf, lof, hif, wf, _ = ci1(gap[1.0])
        if lo > 0: verdict = "quantum BETTER with little data"
        elif hi < 0: verdict = "quantum WORSE with little data"
        else: verdict = f"no advantage detected; at most {hi:+.3f} AUC possible"
        dead = learn[(learn.source == s) & learn.frac.isin(small_fracs) & ~learn.above_chance].frac.tolist()
        out.append({"features": SRC[s], "quantum head": MOD[q],
                    "Δ AUC, little data [95% CI]": f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}]", "wins": f"{w}/{n}",
                    "Δ AUC, all data [95% CI]": f"{mf:+.3f} [{lof:+.3f}, {hif:+.3f}]", "verdict": verdict,
                    "sizes where nothing learns": ", ".join(f"1/{int(round(1 / f))}" for f in dead) or "none"})
T10 = pd.DataFrame(out)
T10.to_csv(f"{OUT}/tables/table10_small_data_test.csv", index=False)
cols = ["features", "quantum head", "Δ AUC, little data [95% CI]", "wins", "Δ AUC, all data [95% CI]"]
with open(f"{OUT}/tables/table10_small_data_test.tex", "w") as fh:
    fh.write("\\begin{table}[htbp]\n\\centering\n\\caption{Quantum minus classical (RBF SVM) test AUC when the classifier is trained on little data (mean over 1/8, 1/4 and 1/2 of the training radiographs) and on all of it. 95\\% confidence intervals from the corrected repeated cross-validation t-test; wins out of 25 splits.}\n\\label{tab:qml_small_data}\n\\small\n")
    fh.write("\\begin{tabular}{llccc}\n\\toprule\n" + " & ".join(c.replace("Δ", "$\\Delta$").replace("%", "\\%") for c in cols) + " \\\\\n\\midrule\n")
    for _, r in T10.iterrows(): fh.write(" & ".join(str(r[c]) for c in cols) + " \\\\\n")
    fh.write("\\bottomrule\n\\end{tabular}\n\\end{table}\n")
print(T10.to_string(index=False))
print()
print("Is the RBF SVM above chance at each size? (if not, comparisons at that size only show that nothing learns)")
print(learn.assign(rbf_auc=learn.rbf_auc.round(3)).pivot_table(index="source", columns="frac", values="rbf_auc").to_string())
nb = (T10.verdict.str.startswith("quantum BETTER")).sum(); nw = (T10.verdict.str.startswith("quantum WORSE")).sum()
print(f"\nSUMMARY: quantum better with little data in {nb} of {len(T10)} feature-set/head combinations; worse in {nw}; "
      f"no detectable difference in {len(T10) - nb - nw}.")